# 01 - EDA clinico

Exploracion de la ficha clinica: distribucion de variables, correlaciones,
perfiles por grupo y proyecciones (PCA/t-SNE). Flujo adaptado del notebook
de clases del profesor (`etapa_1/eda_data_fisiologica.ipynb`), usando los
modulos de `src/` en vez de rutas escritas a mano.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import parallel_coordinates
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src import config, data_io, plotting

In [2]:
plotting.apply_plot_style()
config.create_project_folders()

## Cargar los datos

In [3]:
metadata = data_io.load_metadata()
df_clinical = data_io.load_clinical_data()

label_col = metadata["label"]
id_col = metadata["col_id"]
categorical_features = metadata["categorical_column"]
numerical_features = metadata["numerical_column"]

df_clinical.head()

,patient_id,age,sex_female,height_cm,weight_kg,bmi,waist_cm,hip_cm,waist_hip_ratio,systolic_bp_mmHg,...,traumatic_brain_injury,family_history_alzheimer,sleep_disorder,chronic_kidney_disease,hearing_loss,asthma,allergy_medication,allergy_food,allergy_environmental,alzheimer
0,P0001,75,1,158.7,69.5,27.6,116.6,136.3,0.855,137,...,1,0,0,0,0,0,0,0,0,1
1,P0002,50,1,156.5,70.3,28.7,122.1,130.5,0.936,99,...,0,0,0,0,0,0,0,0,0,1
2,P0003,82,0,169.4,76.6,26.7,117.8,118.3,0.996,153,...,0,1,1,0,0,0,0,1,0,1
3,P0004,72,1,165.2,80.8,29.6,110.1,123.4,0.892,121,...,0,1,0,0,0,0,0,0,1,1
4,P0005,64,1,151.0,75.4,33.0,127.2,144.4,0.881,153,...,0,1,1,0,0,0,0,0,1,0


In [4]:
data_io.save_table(df_clinical.describe(), "clinical_summary.csv")

## Variables numericas

Histograma, boxplot y violinplot por variable, separados por diagnostico.

In [5]:
for column in numerical_features:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    sns.histplot(data=df_clinical, x=column, hue=label_col, ax=axes[0], fill=False, kde=True)
    sns.boxplot(data=df_clinical, x=column, hue=label_col, ax=axes[1])
    sns.violinplot(data=df_clinical, x=column, hue=label_col, ax=axes[2])

    fig.suptitle(column)
    fig.tight_layout()
    data_io.save_figure(fig, f"numerical_{column}.png")
    plt.close(fig)

## Variables categoricas

Conteo por variable binaria, separado por diagnostico.

In [ ]:
def plot_bar(df, column, label_col, filename):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.countplot(data=df, x=column, hue=label_col, ax=ax)
    ax.set_title(column)
    fig.tight_layout()
    data_io.save_figure(fig, filename)
    plt.close(fig)

In [7]:
for column in categorical_features:
    plot_bar(df_clinical, column, label_col, f"categorical_{column}_count.png")

## Correlaciones

In [8]:
df_corr_all = df_clinical[numerical_features].corr()
df_corr_controls = df_clinical[df_clinical[label_col] == 0][numerical_features].corr()
df_corr_cases = df_clinical[df_clinical[label_col] == 1][numerical_features].corr()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
sns.heatmap(df_corr_all, annot=True, fmt=".1f", cmap="Blues", ax=axes[0])
sns.heatmap(df_corr_controls, annot=True, fmt=".1f", cmap="Blues", ax=axes[1])
sns.heatmap(df_corr_cases, annot=True, fmt=".1f", cmap="Blues", ax=axes[2])
axes[0].set_title("Todos")
axes[1].set_title("Controles")
axes[2].set_title("Casos")
fig.tight_layout()
data_io.save_figure(fig, "correlation_full.png")
plt.close(fig)

In [9]:
pairplot = sns.pairplot(df_clinical[numerical_features + [label_col]], hue=label_col)
data_io.save_figure(pairplot.figure, "correlation_pairplot.png")
plt.close(pairplot.figure)

## Comparacion de perfiles (radar)

In [ ]:
def radar_plot(df, columns, label_col, filename, normalize=False):
    data = df[columns].copy()

    if normalize:
        for column in columns:
            min_value = data[column].min()
            max_value = data[column].max()
            data[column] = (data[column] - min_value) / (max_value - min_value)

    data[label_col] = df[label_col].values

    angles = np.linspace(0, 2 * np.pi, len(columns), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": "polar"})

    for value in sorted(data[label_col].unique()):
        subset = data[data[label_col] == value]
        values = [subset[column].mean() for column in columns]
        values += values[:1]
        ax.plot(angles, values, label=str(value))
        ax.fill(angles, values, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(columns, fontsize=8)
    ax.legend(title=label_col)
    fig.tight_layout()
    data_io.save_figure(fig, filename)
    plt.close(fig)

In [11]:
radar_plot(df_clinical, categorical_features, label_col, "radar_categorical.png")
radar_plot(df_clinical, numerical_features, label_col, "radar_continuous.png", normalize=True)

## Parallel coordinates

In [ ]:
def plot_parallel_numeric(df, columns, label_col, filename, sample_per_class=200):
    data = df[columns].copy()
    for column in columns:
        min_value = data[column].min()
        max_value = data[column].max()
        data[column] = (data[column] - min_value) / (max_value - min_value)
    data[label_col] = df[label_col].astype(str).values

    samples = []
    for value in data[label_col].unique():
        group = data[data[label_col] == value]
        samples.append(group.sample(min(len(group), sample_per_class), random_state=42))
    sample = pd.concat(samples)

    fig, ax = plt.subplots(figsize=(12, 6))
    parallel_coordinates(sample, label_col, ax=ax, alpha=0.2)
    fig.tight_layout()
    data_io.save_figure(fig, filename)
    plt.close(fig)

In [13]:
plot_parallel_numeric(df_clinical, numerical_features, label_col, "parallel_coordinates.png")

## PCA y t-SNE

In [ ]:
features = df_clinical.drop(columns=[label_col, id_col])

pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=42)),
])
pca_result = pca_pipeline.fit_transform(features)
df_pca = pd.DataFrame(pca_result, columns=["pca_1", "pca_2"])
df_pca[label_col] = df_clinical[label_col].values

tsne_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("tsne", TSNE(n_components=2, random_state=42)),
])
tsne_result = tsne_pipeline.fit_transform(features)
df_tsne = pd.DataFrame(tsne_result, columns=["tsne_1", "tsne_2"])
df_tsne[label_col] = df_clinical[label_col].values

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(data=df_pca, x="pca_1", y="pca_2", hue=label_col, ax=axes[0])
sns.scatterplot(data=df_tsne, x="tsne_1", y="tsne_2", hue=label_col, ax=axes[1])
axes[0].set_title("PCA")
axes[1].set_title("t-SNE")
fig.tight_layout()
data_io.save_figure(fig, "pca_tsne.png")
plt.close(fig)

## Fin

EDA clinico terminado. Figuras en `results/figures/`, tabla resumen en
`results/tables/clinical_summary.csv`.